* 시작하기 전에 — 작업 폴더 맞추기

In [ ]:

import os        
import pathlib   

here = pathlib.Path.cwd()    

ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)                 

SANDBOX = ROOT / "sandbox" / "w3" / "day02"   

print("프로젝트 루트 :", ROOT)
print("연습 폴더     :", SANDBOX)


httpx2 라이브러리 설치
```
python -m pip install "httpx2==2.12.0"
```

httpx2 사용 예시

In [ ]:
import httpx2

with httpx2.Client(base_url="http://127.0.0.1:8000", timeout=10.0) as client:
    r = client.get("/api/v1/documents", params={"limit": 8})
    r.raise_for_status()         
    documents = r.json()

`frontend/core/api_client.py`

In [ ]:
from __future__ import annotations

import os
from typing import Any

import httpx2

BASE_URL = os.environ.get("API_BASE_URL", "http://127.0.0.1:8000")

TIMEOUT = 10.0

class ApiError(RuntimeError):
    pass

def _request(
    method: str,
    path: str,
    *,
    params: dict | None = None,
    json: dict | None = None,
    emp_no: str | None = None,
) -> Any:

    clean_params = None
    if params is not None:
        clean_params = {k: v for k, v in params.items() if v not in (None, "", "전체")}

    headers = {"X-Emp-No": emp_no} if emp_no else None
    url = f"{BASE_URL}{path}"

    try:
        response = httpx2.request(
            method, url, params=clean_params, json=json, headers=headers, timeout=TIMEOUT
        )
    except httpx2.ConnectError as exc:
        raise ApiError(
            f"백엔드에 연결하지 못했습니다. 터미널에서 서버가 떠 있는지 확인하세요 ({BASE_URL})."
        ) from exc
    except httpx2.TimeoutException as exc:
        raise ApiError("응답이 너무 늦습니다. 서버가 멎었는지 확인하세요.") from exc

    if response.status_code >= 400:
        try:
            message = response.json().get("message") or response.text
        except ValueError:
            message = response.text
        raise ApiError(message)

    return response.json()


def login(emp_no: str, password: str) -> dict:
    pass


def me(emp_no: str) -> dict:
    pass


def list_documents(
    *,
    dept_id: str | None = None,
    security_level: str | None = None,
    status: str | None = None,
    q: str | None = None,
    limit: int = 20,
    emp_no: str | None = None,
) -> list[dict]:
    pass


def get_document(doc_id: str, *, emp_no: str | None = None) -> dict:
    pass


def stats(*, emp_no: str | None = None) -> dict:
    pass


> ✅ **이렇게 나오면 성공입니다**
>
> 터미널 1 을 띄우지 않은 채로 돌렸을 때 —
>
> ```text
> DB 준비 : {'departments': 6, 'users': 7, 'documents': 7, 'versions': 8}
> 노트북 전용 백엔드를 띄웠습니다 : http://127.0.0.1:8000
> 살아 있는가 : True
> ```
>
> 터미널 1 을 이미 띄워 두었다면 첫 두 줄 대신 `백엔드가 이미 떠 있습니다 : http://127.0.0.1:8000`
> 한 줄이 나옵니다. **둘 다 정상입니다.**
> 어제 만든 `app.db` 가 이미 있으면 `DB 준비` 줄의 숫자는 그대로 다시 나옵니다 —
> 시드는 「이미 들어 있으면 아무것도 하지 않는다」로 만들어 두었습니다(9/9).

In [ ]:
# ── 화면을 띄우기 전에 api_client 를 함수 단위로 먼저 확인한다 ──────────
# 화면에서 처음 만나면 「Streamlit 이 이상한가, 내 함수가 이상한가」를 가릴 수 없다.
import sys

# frontend/ 를 모듈 검색 경로 맨 앞에 넣는다. frontend/app.py 가 하는 일과 같다.
# backend 는 넣지 않는다 — 화면 쪽 코드는 backend 를 import 하지 않는다(1절 반례).
FRONTEND = str(ROOT / "frontend")
if FRONTEND not in sys.path:
    sys.path.insert(0, FRONTEND)

from core import api_client   # noqa: E402  (경로를 손본 뒤라야 import 가 된다)

# 8501 이 아니라 8000 인지 눈으로 확인한다. 여기서 매년 사고가 난다.
print("BASE_URL :", api_client.BASE_URL)

try:
    documents = api_client.list_documents(limit=8)
    print("받은 건수 :", len(documents))
    print("첫 건     :", documents[0]["doc_id"], documents[0]["title"])
    print("지표      :", api_client.stats())
except api_client.ApiError as exc:
    # 백엔드가 안 떠 있으면 여기로 온다. 스택 추적 30줄이 아니라 한 문장이 찍힌다 —
    # 3절에서 그 문장을 만들어 둔 것이 이 자리에서 값어치를 한다.
    print("ApiError :", exc)


### 오늘 화면에 뜨는 여덟 줄 — 값의 원본

| `doc_id` | 제목 | 버전 | 부서 | 보안등급 | 형식 | 상태 |
|---|---|---|---|---|---|---|
| `DOC-HR-014` | 국내출장 여비 규정 | `v2.0` | 인사총무 | `일반` | `docx` | 현행 |
| `DOC-HR-014` | 국내출장 여비 규정 | `v1.1` | 인사총무 | `일반` | `docx` | 만료 |
| `DOC-HR-002` | 복무 규정 | `v3.1` | 인사총무 | `일반` | `pdf` | 현행 |
| `DOC-HR-021` | 재택근무 운영 지침 | `v1.2` | 인사총무 | `일반` | `docx` | 현행 |
| `DOC-PU-007` | 구매·계약 규정 | `v4.0` | 구매팀 | `대외비` | `pdf` | 현행 |
| `DOC-SE-003` | 정보보안 지침 | `v2.2` | 보안팀 | `대외비` | `pdf` | 현행 |
| `DOC-SE-011` | 장비 반출입 절차 | `v1.0` | 보안팀 | `3급` | `docx` | 현행 |
| `DOC-PM-005` | 프로젝트 관리 지침 | `v2.0` | PMO | `일반` | `pdf` | 현행 |



상태 배지

| 글자 | 색 | 어디에 |
|---|---|---|
| 현행 · 완료 · 승인 | 초록 (`ag-badge--ok`) | 상태 · 색인 |
| 대기 · 재임베딩 | 주황 (`ag-badge--wait`) | 색인 |
| 만료 · 반려 | 빨강 (`ag-badge--no`) | 상태 |
| 보관 | 회색 (`ag-badge--neutral`) | 색인 |


> 🎙️ **대본**
>
> 여기서 5분만 쉬겠습니다. 표까지 그렸으니 절반은 넘었어요.
>
> 쉬시기 전에 방금 그 여덟 줄, 한 번 눈에 담아 두세요.
> 오후 네 시 반쯤에 우리가 데이터베이스를 통째로 갈아 끼웁니다.
> 그러고 나서 이 화면을 다시 열 건데, **똑같은 여덟 줄이 나와야** 성공입니다.
>
> 한 줄이라도 다르면 뭔가 잘못된 거예요. 그래서 지금 눈에 담아 두시라는 겁니다.
>
> 돌아오시면 필터를 붙입니다. 아까 오전에 배운 세션 상태, 거기서 진짜로 쓰게 됩니다.


`frontend/views/documents.py`

In [ ]:
from __future__ import annotations

from html import escape

import streamlit as st

from core import api_client, session
from ui.badge import badge_html
from ui.metric import metrics
from ui.table import table


DEPTS: dict[str, str | None] = {
    "전체": None,
    "인사총무": "HRGA",
    "구매팀": "PU",
    "보안팀": "SE",
    "PMO": "PMO",
}

LEVELS = ["전체", "일반", "3급", "대외비"]

STATUSES = ["전체", "현행", "만료"]

HEADERS = ["문서 ID", "문서명", "버전", "시행 ~ 만료", "상태", "부서", "등급", "색인"]

ALIGNS = ["ag-nowrap", "", "", "ag-nowrap", "", "", "", "ag-nowrap"]


def _metrics_row() -> None:
    try:
        counts = api_client.stats(emp_no=session.emp_no())
    except api_client.ApiError as exc:
        st.caption(f"지표를 불러오지 못했습니다: {exc}")
        return

    metrics([
        {"label": "전체", "value": counts["total"], "delta": "문서 버전 기준"},
        {"label": "현행", "value": counts["current"], "delta": "지금 유효한 판",
         "tone": "ok"},
        {"label": "만료", "value": counts["expired"], "delta": "지난 판",
         "tone": "no"},
        {"label": "재임베딩", "value": counts["reindexing"], "delta": "색인을 다시 만드는 중",
         "tone": "wait"},
    ])


def _filter_row() -> dict:
    left, middle, right, search = st.columns([1, 1, 1, 2])

    with left:
        dept_name = st.selectbox("부서", list(DEPTS), key="f_dept")
    with middle:
        level = st.selectbox("보안등급", LEVELS, key="f_level")
    with right:
        status = st.selectbox("상태", STATUSES, key="f_status")
    with search:
        keyword = st.text_input("검색어", key="f_q", placeholder="문서명 또는 문서 ID")

    return {
        "dept_id": DEPTS[dept_name],
        "security_level": None if level == "전체" else level,
        "status": None if status == "전체" else status,
        "q": keyword or None,
    }


def _table(documents: list[dict]) -> None:
    rows = []
    for document in documents:
        period = f"{document['effective_from']} ~ {document['expires_at'] or '현행'}"
        index_label = f"{document['index_status']} {document['index_progress']}%"
        rows.append([
            escape(document["doc_id"]),
            escape(document["title"]),
            escape(document["version"]),
            escape(period),
            badge_html(document["status"]),
            escape(document["dept"]),
            escape(document["security_level"]),
            badge_html(index_label),
        ])

    table(HEADERS, rows, align=ALIGNS)


def render() -> None:
    st.title("문서 관리")
    st.caption("상태 필터가 「전체」라 지난 판까지 함께 보입니다. "
               "「현행」으로 좁히면 현재 유효한 최신본만 남습니다.")

    _metrics_row()

    filters = _filter_row()

    if st.button("새 문서 업로드"):
        st.info("업로드 화면은 W4 에 만듭니다.")

    with st.spinner("문서를 불러오는 중입니다..."):
        try:
            documents = api_client.list_documents(**filters, limit=100,
                                                  emp_no=session.emp_no())
        except api_client.ApiError as exc:
            st.error(str(exc))
            return

    if not documents:
        st.info("조건에 맞는 문서가 없습니다. 필터를 바꿔 보세요.")
        return

    st.caption(f"{len(documents)}건")
    _table(documents)


`frontend/app.py`

In [ ]:
from __future__ import annotations

import html
import pathlib
import sys

import streamlit as st

sys.path.insert(0, str(pathlib.Path(__file__).parent))

from core import router, session 
from ui.theme import inject_css  
from views import documents as documents_view 
from views import login as login_view  

st.set_page_config(
    page_title="사내 업무 에이전트",
    layout="wide",
    initial_sidebar_state="expanded",
)

inject_css()

NAV: list[tuple[str, str | None]] = [
    ("AI 업무 도우미", None),      
    ("문서 관리", "documents"),    
    ("승인함", None),             
    ("운영 대시보드", None),       
]

def render_sidebar() -> None:
    st.sidebar.markdown(
        '<div class="ag-brand"><div class="ag-brand-name">사내 업무 에이전트</div></div>',
        unsafe_allow_html=True,
    )

    user = session.current_user()
    if user is not None:
        st.sidebar.markdown(
            '<div class="ag-user"><div>'
            f'<div class="ag-user-name">{html.escape(user["name"])}</div>'
            f'<div class="ag-user-role">{html.escape(user["dept"])}</div>'
            '</div></div>',
            unsafe_allow_html=True,
        )
        if st.sidebar.button("로그아웃", key="nav_logout"):
            session.logout()


    for label, page_key in NAV:
        if st.sidebar.button(label, key=f"nav_{label}"):
            if page_key is None:
                st.sidebar.info("아직 만들지 않은 화면입니다.")
            else:
                st.session_state["page"] = page_key


def main() -> None:
    session.init_state()

    if not session.is_authenticated():
        login_view.render()
        return

    render_sidebar()

    page = router.current_page()
    if page == "documents":
        documents_view.render()
    else:
        st.info("아직 만들지 않은 화면입니다.")


main()
